## Использование линейной регрессии для построения причинно-следственных выводов

**Материал подготовлен для курса [Объяснимое машинное обучение](https://ai-interpretability.school).**

Стараясь оценить связь признака с целевой переменной, исследовательницы часто прибегают к анализу коэффициента корреляции. Кроме того, коэффициент корреляции — это первое, с чем знакомят на курсах Data science, так что знание о нем есть даже у начинающих. Однако:

> *Correlation is not causation.*

или «корреляция не равна причинно-следственной связи». Но как быть, если причинно-следственную связь оценить все же хочется?

В модуле курса «Линейные интерпретируемые модели» коэффициенты регрессии служат для анализа важности признаков. Здесь мы посмотрим, когда коэффициент регрессии можно читать как причинный эффект, а когда нельзя, и чем его дополняют: контролем по наблюдаемым переменным, DML и инструментальной переменной.

Чтобы материал был понятен каждому, начнем с самого начала.

In [ ]:
#Импорт библиотек
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_predict

Оценим, как готовность получить высшее образование связана с оценкой по математике.

**Математика**

Пусть:\
$T$ — признак, влияние которого мы хотим оценить: готовность получить высшее образование ($T=1$, если студентка готова его получать, $T=0$, если нет)\
$Y_0$ — оценка студентки, если она не готова получать высшее образование\
$Y_1$ — оценка студентки, если она готова получать высшее образование\
$Y$ — наблюдаемая оценка: $Y = T\,Y_1 + (1-T)\,Y_0$, то есть $Y=Y_1$ при $T=1$ и $Y=Y_0$ при $T=0$\
$TE = Y_1 - Y_0$ — эффект воздействия: насколько меняется оценка при переходе признака с 0 на 1

In [ ]:
#В идеальном мире наш набор данных выглядел бы так (данные придуманы для примера)
ate_att_data = pd.DataFrame(dict(
      i= [1,2,3,4],
      Y0=[500,600,800,700],
      Y1=[450,600,600,750],
      T= [0,0,1,1],
      Y= [500,600,600,750],
      TE=[-50,0,-200,50],
))
ate_att_data

In [ ]:
#Но в реальном он выглядит так: у каждой студентки известен только один из двух исходов
pd.DataFrame(dict(
    i= [1,2,3,4],
    Y0=[500,600,np.nan,np.nan],
    Y1=[np.nan,np.nan,600,750],
    T= [0,0,1,1],
    Y= [500,600,600,750],
    TE=[np.nan,np.nan,np.nan,np.nan],
))

Студенток у нас много. Для удобства введем еще пару вещей:

- $Y_{i0}$ — потенциальная оценка $i$-й студентки при неготовности получать высшее образование
- $Y_{i1}$ — потенциальная оценка $i$-й студентки при готовности получать высшее образование

- Тогда влияние признака «готовность получить высшее образование» можно определить как: $Y_{i1} - Y_{i0}$


- $ATE$, *average treatment effect*, — средний эффект наличия признака
- $ATT$, *average treatment effect on treated*, — средний эффект наличия признака для наблюдений, у которых он есть

Заметим, что:\
$ATE = E[Y_1 - Y_0]$ — матожидание от разности целевого значения с наличием и отсутствием признака\
$ATT = E[Y_1 - Y_0|T=1]$ — матожидание от разности целевого значения с наличием и отсутствием признака при условии, что признак равен 1.

По идеальному набору данных мы смогли бы рассчитать все по определению. По реальному — нет. И так как мы решили работать с реальностью, рассмотрим:

$$E[Y|T=1] - E[Y|T=0] = E[Y_1|T=1] - E[Y_0|T=0] = E[Y_1|T=1] - E[Y_0|T=1] + E[Y_0|T=1] - E[Y_0|T=0]=E[Y_1-Y_0|T=1] + (E[Y_0|T=1] - E[Y_0|T=0])$$


Получили: $E[Y_1-Y_0|T=1] + (E[Y_0|T=1] - E[Y_0|T=0])$

Первое слагаемое $E[Y_1-Y_0|T=1]$ — это ATT, средний эффект воздействия на воздействованных: то, что мы и хотим измерить. Второе $(E[Y_0|T=1] - E[Y_0|T=0])$ — селекционное смещение: разница между группами в отсутствие воздействия. Оно систематическое, а не случайное, и именно его обнуляет рандомизация.

**Постановка проблемы:** \
Влияние некоторых переменных невозможно проверить, используя A/B-тестирование. Например:

- Некоторые эксперименты неэтичны. Нельзя заставить студенток употреблять алкоголь, чтобы увидеть, как это повлияет на их успеваемость в университете.
- Проведение ряда других экспериментов запрещено юридическими ограничениями. Например, вы не можете устанавливать разные цены на один и тот же продукт.
- Также существуют ситуации, когда провести эксперимент невозможно. Если происходит ребрендинг, запускать товар на две достаточно большие непересекающиеся группы может быть дорого.


В этом случае исследовательнице приходится работать с тем, что есть.

Что может помочь оценить влияние признака? **Линейная регрессия!**

**Используемый датасет:** [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance) — анкеты и оценки 395 учащихся двух португальских школ; итоговая оценка по математике `G3` стоит по шкале от 0 до 20.


Выясним, как готовность получить высшее образование влияет на итоговую оценку по математике. А именно попробуем построить модель:

$$\text{math\_grade} = \beta_0 + \beta_1 \cdot \text{higher} + \varepsilon,$$
где:
- $\beta_0$ — свободный член
- $\beta_1$ — коэффициент при признаке: величина, на которую меняется прогноз оценки, когда готовность получить высшее образование переходит с 0 на 1
- $\text{higher}$ — готовность получить высшее образование
- $\varepsilon$ — случайный шум

In [ ]:
data = pd.read_csv('https://github.com/aiedu-courses/all_datasets/raw/main/student-mat.csv',
                   sep=';')
data.head()

In [ ]:
model = smf.ols('G3 ~ higher', data=data).fit()
model.summary().tables[1]

**Что значит каждый коэффициент?**
- coef — при Intercept значение целевой переменной, когда все признаки равны 0; при признаке — изменение прогноза, когда признак переходит с 0 на 1;
- std err — стандартная ошибка оценки коэффициента;
- P>|t| — p-value: так как здесь оно меньше 0.05, можно сказать, что готовность к высшему образованию значимо связана с прогнозом.

Но на этом информация из регрессии не заканчивается!

In [ ]:
model.summary()

Здесь:
- Когда была обучена модель;
- Количество наблюдений;
- Метод обучения OLS;
- Качество регрессии: $R^2, R^2_{adj}$;
- F-statistic: значение статистики теста Фишера с нулевой гипотезой «регрессия не значима», то есть совсем не информативна;
- Prob (F-statistic): p-value этого теста — вероятность получить такое же или большее значение F-статистики, если регрессия неинформативна;
- Log-Likelihood: логарифм правдоподобия по выборке;
- AIC, BIC: информационные критерии для сравнения регрессий.

## Контроль по наблюдаемым переменным

Коэффициент $\hat\beta_1$ модели с одним признаком — это разность средних оценок двух групп, а в ней, как мы видели выше, смешаны эффект и селекционное смещение. Если учащиеся, которые собираются получать высшее образование, отличаются от остальных чем-то еще — временем на учебу, числом прошлых неудач, образованием родителей, — эта разница попадет в $\hat\beta_1$.

Первое средство — добавить такие переменные в регрессию как контроли. Возьмем все анкетные признаки, кроме промежуточных оценок `G1` и `G2`: их ставят в том же учебном году, они сами зависят от готовности учиться дальше, и контроль по ним отрезал бы часть эффекта, который мы хотим измерить (это медиаторы).

In [ ]:
controls = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob',
            'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid',
            'activities', 'nursery', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
            'Walc', 'health', 'absences']
print(len(controls), 'контролей')
print(data['higher'].value_counts())

**Задание 1. Постройте OLS-модель `G3 ~ higher + <все контроли из списка controls>` и выведите коэффициент при `higher[T.yes]`, его стандартную ошибку и p-value. Как изменился коэффициент по сравнению с моделью с одним признаком?**

In [ ]:
# Ваш код здесь

## DML: остатки вместо формулы

В линейной регрессии с контролями мы сами решаем, в каком виде контроли входят в модель: линейно, без взаимодействий. DML (double/debiased machine learning, [Chernozhukov et al., 2018](https://arxiv.org/abs/1608.00060)) снимает это ограничение. Базовый сценарий — partial linear regression:

$$Y = \theta\,T + g(X) + \varepsilon,\qquad T = m(X) + \nu.$$

1. Любой ML-моделью предсказываем $E[Y\mid X]$ и $E[T\mid X]$.
2. Считаем остатки $\tilde Y = Y - \hat E[Y\mid X]$ и $\tilde T = T - \hat m(X)$.
3. Оценка эффекта — коэффициент регрессии остатков: $\hat\theta = \sum_i \tilde T_i \tilde Y_i \big/ \sum_i \tilde T_i^2$.

Чтобы переобучение вспомогательных моделей не попало в оценку, прогнозы для каждого объекта делаются моделями, которые его не видели: это cross-fitting. Ниже матрица признаков `X` собрана из тех же контролей.

In [ ]:
X = pd.get_dummies(data[controls], drop_first=True).astype(float).values
y = data['G3'].values.astype(float)
t = (data['higher'] == 'yes').astype(float).values
print(X.shape)

**Задание 2. Реализуйте DML с cross-fitting на 5 фолдах (`KFold(5, shuffle=True, random_state=0)`): $E[Y\mid X]$ — `RandomForestRegressor(n_estimators=500, min_samples_leaf=5, random_state=0)`, $E[T\mid X]$ — `RandomForestClassifier` с теми же настройками и `predict_proba`. Выведите $\hat\theta$, его стандартную ошибку и долю объектов, у которых $\hat m(X) > 0.95$.**

Стандартная ошибка DML: $\widehat{se} = \sqrt{\tfrac1n\sum_i \tilde T_i^2\,(\tilde Y_i-\hat\theta\tilde T_i)^2\big/n}\;\Big/\;\tfrac1n\sum_i \tilde T_i^2$.

Оформите решение функцией `dml(y, t, X)`, она понадобится еще раз.

In [ ]:
# Ваш код здесь

Посмотрите на долю объектов с $\hat m(X)$ около единицы. Не собираются получать высшее образование только 20 учащихся из 395, и для большинства остальных модель склонности уверена, что они в группе $T=1$. У таких объектов остаток $\tilde T$ почти нулевой, и оценка $\hat\theta$ держится на немногих объектах. Это нарушение позитивности, о котором говорит урок «IV и DML»: DML аккуратно считает эффект при условии, что конфаундеры измерены все и что в каждой области признаков есть объекты обеих групп.

## Инструментальная переменная

DML и регрессия с контролями убирают смещение только от тех конфаундеров, которые измерены. Если есть скрытый конфаундер — например, мотивация, которой в анкете нет, — нужен другой источник вариации воздействия: инструментальная переменная $Z$. Она должна влиять на воздействие (relevance), не влиять на исход иначе как через воздействие (exclusion) и не зависеть от скрытых конфаундеров (independence).

В данных UCI Student Performance такого инструмента нет. Признаки, которые связаны с `higher`, — образование родителей, платные занятия, прошлые неудачи — влияют на оценку и напрямую, то есть нарушают exclusion, а проверить exclusion по данным нельзя.

Поэтому IV посчитаем на классическом наборе данных для этого метода — данных [Mroz (1987)](https://doi.org/10.2307/1911029) о 753 замужних женщинах США в 1975 году; 428 из них в тот год работали, и у них записана зарплата. Набор входит в учебник Wooldridge «Introductory Econometrics» и в пакет [`wooldridge`](https://github.com/spring-haru/wooldridge); копия лежит в папке `data` репозитория курса. Вопрос — тот же, что в уроке «IV и DML»: эффект образования на зарплату.

- исход $Y$ — логарифм часовой зарплаты `lwage`;
- воздействие $T$ — число лет обучения `educ`;
- контроли — опыт работы в годах `exper` и его квадрат `expersq`;
- инструменты $Z$ — число лет обучения матери `motheduc` и отца `fatheduc`.

Довод за такой инструмент: дети более образованных родителей учатся дольше (relevance), а на зарплату образование родителей, по допущению, влияет только через образование самой женщины (exclusion).

In [ ]:
mroz = pd.read_csv('https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/mroz.csv')
work = mroz[mroz['inlf'] == 1].copy()
print(len(mroz), 'женщин, из них работали', len(work))
work[['lwage', 'educ', 'exper', 'motheduc', 'fatheduc']].describe().round(2)

**Задание 3. Сравните OLS и 2SLS на данных Mroz для работавших женщин.**

1. **OLS:** `lwage ~ educ + exper + expersq`; выведите коэффициент при `educ` и его стандартную ошибку.
2. **Первая стадия:** OLS `educ ~ motheduc + fatheduc + exper + expersq`; выведите F-статистику совместной значимости двух инструментов, `first.f_test('motheduc = 0, fatheduc = 0')`. Сравните ее с порогом 10 из урока «IV и DML».
3. **Вторая стадия:** OLS `lwage ~ educ_hat + exper + expersq`, где `educ_hat` — прогноз первой стадии. Коэффициент при `educ_hat` — оценка 2SLS.

Исход стоит в логарифме, поэтому коэффициент при `educ` читается как доля: 0.05 значит, что год обучения поднимает зарплату примерно на 5 %.

*Стандартная ошибка второй стадии, сделанной вручную, неверна: остатки в ней посчитаны по `educ_hat`, а должны — по настоящему `educ`. Чтобы ее исправить, посчитайте остатки $Y - X\hat\beta$ с исходной матрицей $X$, где стоит `educ`, их дисперсию $\hat\sigma^2$ на `second.df_resid` степеней свободы и умножьте стандартную ошибку второй стадии на $\sqrt{\hat\sigma^2/\texttt{second.scale}}$. Готовая реализация с верной ошибкой — `IV2SLS` из пакета `linearmodels`.*

In [ ]:
# Ваш код здесь

**Три условия инструмента на этих данных.**

- **Relevance** проверяется по данным: F-статистика первой стадии сравнивается с порогом 10 ([Staiger, Stock, 1997](https://doi.org/10.2307/2171753)).
- **Exclusion** по данным не проверить. Образование родителей может влиять на зарплату женщины и в обход ее собственного образования: через способности, которые передаются в семье, через связи, выбор профессии и места работы. Если такой путь есть, 2SLS тоже смещена.
- **Independence** — образование родителей должно быть «как будто случайным» относительно всего, что поднимает зарплату дочери, кроме ее образования. Это тоже допущение, а не результат расчета.

Поэтому разница между OLS и 2SLS здесь не доказывает, что OLS завышает эффект образования: она показывает, что дают эти данные, если принять допущения об образовании родителей. Стандартная ошибка 2SLS больше, чем у OLS, потому что оценка использует только ту часть вариации образования, которую объясняют инструменты.

## Когда истинный эффект известен: полусинтетические данные

На настоящих данных истинного эффекта не знает никто, и проверить, какая оценка ближе к нему, нельзя. Чтобы увидеть, что делают со скрытым конфаундером регрессия, IV и DML, возьмем полусинтетические данные: признаки учащихся UCI Student Performance настоящие, а воздействие, инструмент и исход сгенерированы так, что истинный эффект известен. Сюжет — как в задаче урока «IV и DML»: школа разыгрывает бесплатные места на подготовительный курс по лотерее ($Z$), курс ($T$) проходят чаще выигравшие и более мотивированные, мотивация ($U$) в данных не записана и сама поднимает оценку по математике. Истинный эффект курса $\theta = 2$ балла.

In [ ]:
rng = np.random.default_rng(42)
n = len(data)
U = rng.normal(size=n)                                   # мотивация: скрытый конфаундер
Z = rng.binomial(1, 0.5, size=n)                         # лотерея: инструмент
g_x = 0.3 * (data['studytime'].values - 2) - 0.5 * data['failures'].values
T_syn = (-0.3 + 1.5 * Z + 1.0 * U + g_x + rng.normal(size=n) > 0).astype(float)
theta_true = 2.0
f_x = 0.8 * data['studytime'].values - 1.5 * data['failures'].values + 0.3 * data['Medu'].values
Y_syn = 8 + theta_true * T_syn + f_x + 2.0 * U + rng.normal(size=n)

syn = pd.DataFrame(X, columns=[f'x{j}' for j in range(X.shape[1])])
syn['Y'], syn['T'], syn['Z'] = Y_syn, T_syn, Z
x_terms = ' + '.join(f'x{j}' for j in range(X.shape[1]))
print('доля прошедших курс:', round(T_syn.mean(), 3))
print('OLS без контролей:', round(smf.ols('Y ~ T', data=syn).fit().params['T'], 3))
print('OLS с контролями: ', round(smf.ols('Y ~ T + ' + x_terms, data=syn).fit().params['T'], 3))

**Задание 4. Посчитайте на полусинтетических данных оценки эффекта курса и сравните их с истинным $\theta = 2$.**

1. **2SLS по шагам:** первый шаг — OLS `T ~ Z + <контроли x_terms>`; выведите F-статистику инструмента (квадрат t-статистики при `Z`). Второй шаг — OLS `Y ~ T_hat + <контроли>`, где `T_hat` — прогноз первого шага.
2. **Оценка Вальда:** разность средних `Y` между выигравшими и проигравшими лотерею, деленная на разность долей прошедших курс.
3. **DML:** функция `dml` из задания 2 дважды — с признаками `X` и с признаками `X` плюс столбец мотивации `U` (как если бы ее измерили).

In [ ]:
# Ваш код здесь

Сравните оценки на полусинтетических данных. Регрессия с контролями и DML по $X$ видят одни и те же переменные и потому одинаково не видят мотивацию: от того, что вспомогательные модели стали гибкими, скрытый конфаундер не становится измеренным. IV использует только ту часть вариации курса, которую создала лотерея, а она от мотивации не зависит. DML возвращается к истинному эффекту, когда мотивация попадает в признаки.

## Насколько сильным должен быть скрытый конфаундер

На полусинтетических данных мотивацию $U$ мы знаем и можем посчитать, сколько она дает. На настоящих данных скрытого конфаундера никто не видел, и вопрос ставят наоборот: насколько сильным он должен быть, чтобы оценка исчезла. Ответ дают три шага урока «IV и DML».

**Теорема Фриша–Во–Ловелла** ([Frisch и Waugh, 1933](https://doi.org/10.2307/1907330); [Lovell, 1963](https://doi.org/10.1080/01621459.1963.10480682)). Коэффициент при $T$ в регрессии $Y$ на $T$ и контроли $X$ равен коэффициенту регрессии остатков $\tilde Y$ на остатки $\tilde T$, где $\tilde Y$ и $\tilde T$ — остатки регрессий $Y$ и $T$ на $X$:

$$\hat\beta_T = \frac{\sum_i \tilde T_i \tilde Y_i}{\sum_i \tilde T_i^2}.$$

DML из задания 2 устроен так же, только остатки в нем считают ML-модели.

**Задание 5. На полусинтетических данных посчитайте остатки регрессий `Y ~ <контроли x_terms>` и `T ~ <контроли x_terms>` и коэффициент регрессии первых остатков на вторые. Сравните его с коэффициентом при `T` в модели `Y ~ T + <контроли>`, которая выше дала OLS с контролями.**

In [ ]:
# Ваш код здесь

**Смещение от пропущенной переменной.** Пусть «длинная» регрессия $Y$ на $T$, $X$ и $U$ дает коэффициент $\hat\beta_{long}$ при $T$ и $\hat\gamma$ при $U$, а «короткая», без $U$, — $\hat\beta_{short}$. Обозначим $\hat\delta$ коэффициент при $T$ во вспомогательной регрессии $U$ на $T$ и $X$. Тогда на любой выборке

$$\hat\beta_{short} - \hat\beta_{long} = \hat\gamma\,\hat\delta:$$

смещение — произведение влияния пропущенной переменной на исход и ее связи с воздействием при тех же контролях. Знак смещения — знак произведения.

**Задание 6. Добавьте мотивацию в таблицу (`syn['U'] = U`), постройте длинную регрессию `Y ~ T + U + <контроли>` и вспомогательную `U ~ T + <контроли>`. Выведите $\hat\gamma$, $\hat\delta$, их произведение и разность $\hat\beta_{short} - \hat\beta_{long}$.**

In [ ]:
# Ваш код здесь

**Анализ чувствительности по Cinelli и Hazlett** ([2020](https://doi.org/10.1111/rssb.12348)). Формулу смещения переписывают через частные $R^2$, которые не зависят от единиц измерения $U$:

- $R^2_{Y\sim U\mid T,X}$ — доля остаточной дисперсии $Y$, которую объясняет $U$ сверх $T$ и $X$;
- $R^2_{T\sim U\mid X}$ — доля остаточной дисперсии $T$, которую объясняет $U$ сверх $X$.

$$|\text{смещение}| = \widehat{se}(\hat\beta_{short})\,\sqrt{df}\,\sqrt{\frac{R^2_{Y\sim U\mid T,X}\;R^2_{T\sim U\mid X}}{1 - R^2_{T\sim U\mid X}}},$$

где $\widehat{se}$ и $df$ — стандартная ошибка и число степеней свободы короткой регрессии. Частный $R^2$ коэффициента считают по его t-статистике: $R^2 = t^2/(t^2 + df)$.

Сводка одним числом — **robustness value**: конфаундер, который объясняет долю $RV_q$ остаточной дисперсии и $T$, и $Y$, уменьшает оценку на $100q\,\%$; при $q=1$ — до нуля.

$$RV_q = \tfrac12\Big(\sqrt{f_q^4 + 4f_q^2} - f_q^2\Big),\qquad f_q = q\,\frac{|t_T|}{\sqrt{df}}.$$

Сначала проверим формулу смещения на полусинтетических данных, где частные $R^2$ мотивации можно посчитать.

In [ ]:
def partial_r2(t_stat, df):
    return t_stat ** 2 / (t_stat ** 2 + df)

def robustness_value(t_stat, df, q=1.0):
    f = q * abs(t_stat) / np.sqrt(df)
    return 0.5 * (np.sqrt(f ** 4 + 4 * f ** 2) - f ** 2)

df_short = short.df_resid
r2_yu = partial_r2(long.tvalues['U'], long.df_resid)
t_on_u = smf.ols('T ~ U + ' + x_terms, data=syn).fit()
r2_tu = partial_r2(t_on_u.tvalues['U'], t_on_u.df_resid)
bias = short.bse['T'] * np.sqrt(df_short) * np.sqrt(r2_yu * r2_tu / (1 - r2_tu))
print(f'R2(Y~U|T,X) = {r2_yu:.3f} | R2(T~U|X) = {r2_tu:.3f} | смещение по формуле {bias:.3f}')
print(f'RV для коэффициента при T: {robustness_value(short.tvalues["T"], df_short):.3f}')

Формула дает ровно $\hat\gamma\hat\delta$ из задания 6. Мотивация объясняет больше остаточной дисперсии $Y$, чем RV, но меньше остаточной дисперсии $T$, поэтому она сдвигает оценку почти вдвое, но до нуля не доводит: длинная регрессия дает около 2.

**Задание 7. Вернемся к настоящим данным UCI Student Performance и модели с контролями из задания 1. Посчитайте частный $R^2$ и robustness value $RV_1$ для коэффициента при `higher[T.yes]`, а также частные $R^2$ признака `failures` (число прошлых неудач): с исходом — в той же модели, с воздействием — в регрессии `higher` (как 0 и 1) на те же контроли. Подставьте их в формулу смещения: сколько осталось бы от оценки, если бы скрытый конфаундер был так же силен, как `failures`?**

In [ ]:
# Ваш код здесь

Оценка эффекта готовности учиться дальше хрупкая: конфаундеру достаточно объяснять несколько процентов остаточной дисперсии и `higher`, и оценки. Один измеренный признак, `failures`, объясняет больше этого в дисперсии оценки и немного меньше — в дисперсии `higher`, и конфаундер такой силы снял бы большую часть оценки. Сравнение с измеренным признаком Cinelli и Hazlett называют benchmarking; в статье его делают аккуратнее, через силу конфаундера в $k$ раз больше силы признака, но вывод тот же: скрытому конфаундеру не обязательно быть экзотическим, чтобы изменить вывод. Полную версию с границами по нескольким признакам считает пакет [PySensemakr](https://github.com/nlapier2/PySensemakr) (лицензия MIT); для DML тот же подход развит в [Chernozhukov et al., 2021](https://arxiv.org/abs/2112.13398) и реализован в `doubleml` методом `sensitivity_analysis`.

**E-value** ([VanderWeele и Ding, 2017](https://doi.org/10.7326/M16-2607)) задан на шкале отношения рисков, а у нас исход непрерывный. Для такого исхода оценку сначала переводят в стандартизованную разность $d = \hat\beta_T/\mathrm{SD}(Y)$ и приближенно — в отношение рисков, как в функции `evalues.OLS` R-пакета [EValue](https://cran.r-project.org/package=EValue):

$$RR \approx e^{0.91\,d},\qquad \big[e^{0.91\,d - 1.78\,se_d},\ e^{0.91\,d + 1.78\,se_d}\big],\qquad se_d = \widehat{se}(\hat\beta_T)/\mathrm{SD}(Y),$$

а затем считают $E = RR + \sqrt{RR\,(RR-1)}$ — для оценки и для границы интервала, ближайшей к единице. Если интервал уже накрывает единицу, E-value интервала равен 1: никакого конфаундера не нужно, чтобы интервал задел «нет эффекта». Деление на $\mathrm{SD}(Y)$, а не на остаточное стандартное отклонение, дает осторожную оценку: $d$ меньше, и E-value меньше.

**Задание 8. Для коэффициента при `higher[T.yes]` из задания 1 посчитайте $d$, приближенное отношение рисков и его интервал, E-value оценки и E-value интервала. $\mathrm{SD}(Y)$ — выборочное стандартное отклонение `G3` (`data['G3'].std()`). Ответ — E-value оценки, округленный до двух знаков после точки.**

In [ ]:
# Ваш код здесь

Чтобы объяснить оценку целиком, скрытому конфаундеру нужно примерно вдвое поднимать и шанс готовности учиться дальше, и шанс высокой оценки. Интервал же накрывает единицу и без конфаундера: E-value интервала равен 1. E-value и robustness value говорят об одном — вывод хрупкий, — но на разных шкалах: E-value — во сколько раз конфаундер меняет вероятности, RV — какую долю остаточной дисперсии он объясняет. E-value удобен, когда в статье есть только оценка и интервал, RV точнее для линейной регрессии.

**Выводы. Подчеркнем, что:**

- Моделирование связей «причина — следствие» ограничено наличием скрытых взаимосвязей в данных. Поэтому к любым причинно-следственным выводам, основанным на наблюдениях, следует относиться с долей скептицизма. Мы не можем быть уверены, что полученная нами оценка полностью объективна.

- Контроль по наблюдаемым переменным и DML убирают смещение только от измеренных конфаундеров; DML вдобавок требует, чтобы в каждой области признаков были объекты обеих групп.

- Инструментальная переменная справляется и со скрытым конфаундером, но держится на допущении exclusion, которое по данным не проверить: на данных Mroz оно означает, что образование родителей влияет на зарплату только через образование самой женщины.

- Имея данные наблюдений, мы можем только выдвигать гипотезы и проверять их на основе моделей.

- Регрессия из statsmodels сохраняет информацию, позволяющую анализировать построенную модель сразу после построения, без дополнительных тестов.

Спасибо за ваше время на материал!

Источники:
1. [Causal Inference for the Brave and True](https://matheusfacure.github.io/python-causality-handbook/01-Introduction-To-Causality.html)
2. [Linear regressions for causal conclusions](https://towardsdatascience.com/linear-regressions-for-causal-conclusions-34c6317c5a11)
3. [Chernozhukov et al., 2018. Double/debiased machine learning for treatment and structural parameters](https://arxiv.org/abs/1608.00060)
4. [Mroz, 1987. The Sensitivity of an Empirical Model of Married Women's Hours of Work to Economic and Statistical Assumptions](https://doi.org/10.2307/1911029)
5. [Staiger, Stock, 1997. Instrumental Variables Regression with Weak Instruments](https://doi.org/10.2307/2171753)